<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

[ваш текст]

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте полиморфизм с перекрытием и прегегрузкой методов, а также generic классы

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [4]:
using System;
using System.Collections.Generic;
using System.Linq;

// Базовый интерфейс для всех подписок
public interface ISubscription
{
    void Activate();
    void DisplayInfo();
}

// Интерфейс для дополнительных возможностей
public interface IEmailNotification
{
    void SendRenewalReminder();
}

public interface IPremiumContent
{
    List<string> ExclusiveContent { get; set; }
    void AccessPremiumFeature();
}

// Generic класс для управления подписками
public class SubscriptionManager<T> where T : Subscription
{
    private List<T> _subscriptions = new List<T>();

    public void AddSubscription(T subscription)
    {
        _subscriptions.Add(subscription);
        Console.WriteLine($"Подписка {subscription.ServiceName} добавлена в менеджер");
    }

    public void RemoveSubscription(string subscriptionId)
    {
        var sub = _subscriptions.FirstOrDefault(s => s.SubscriptionId == subscriptionId);
        if (sub != null)
        {
            _subscriptions.Remove(sub);
            Console.WriteLine($"Подписка {sub.ServiceName} удалена из менеджера");
        }
    }

    public void DisplayAllSubscriptions()
    {
        Console.WriteLine($"\n=== Все подписки в менеджере ({typeof(T).Name}) ===");
        foreach (var sub in _subscriptions)
        {
            sub.GetSubscriptionDetails();
            Console.WriteLine("---");
        }
    }

    public T GetSubscriptionById(string id)
    {
        return _subscriptions.FirstOrDefault(s => s.SubscriptionId == id);
    }
}

// Базовый класс для всех подписок
public class Subscription : ISubscription
{
    // Новые атрибуты
    public DateTime StartDate { get; set; }
    public DateTime EndDate { get; set; }
    public bool IsActive { get; set; }
    public int MinSubscriptionPeriod { get; set; } = 1;
    public string Description { get; set; } // Новый атрибут
    public int PriorityLevel { get; set; } // Новый атрибут
    public bool AutoRenewal { get; set; } // Новый атрибут
    public int TotalMonthsSubscribed { get; set; } // Новый атрибут

    // Существующие атрибуты
    public string SubscriptionId { get; set; }
    public string ServiceName { get; set; }
    public decimal Cost { get; set; }

    // Конструкторы (перегрузка методов)
    public Subscription(string id, string name, decimal cost)
    {
        SubscriptionId = id;
        ServiceName = name;
        Cost = cost;
        StartDate = DateTime.Now;
        EndDate = StartDate.AddMonths(1);
        IsActive = true;
        Description = "Базовая подписка";
        PriorityLevel = 1;
        AutoRenewal = false;
        TotalMonthsSubscribed = 0;
    }

    public Subscription(string id, string name, decimal cost, string description) : this(id, name, cost)
    {
        Description = description;
    }

    public Subscription(string id, string name, decimal cost, string description, int priority) : this(id, name, cost, description)
    {
        PriorityLevel = priority;
    }

    // Новые методы
    public void Deactivate()
    {
        IsActive = false;
        Console.WriteLine("Подписка деактивирована.");
    }

    public virtual void Activate()
    {
        IsActive = true;
        StartDate = DateTime.Now;
        EndDate = StartDate.AddMonths(1);
        Console.WriteLine("Подписка активирована.");
    }

    public virtual int GetRemainingDays()
    {
        return (EndDate - DateTime.Now).Days;
    }

    // Перегруженные методы
    public virtual void ExtendSubscription(int months = 1)
    {
        EndDate = EndDate.AddMonths(months);
        TotalMonthsSubscribed += months;
        Console.WriteLine($"Подписка {ServiceName} продлена на {months} месяц(ев)");
    }

    public virtual void ExtendSubscription(int months, bool applyDiscount)
    {
        if (applyDiscount && months >= 3)
        {
            decimal discount = Cost * months * 0.1m;
            Console.WriteLine($"Применена скидка 10%: -{discount:C}");
        }
        ExtendSubscription(months);
    }

    // Существующие методы
    public virtual decimal CalculateMonthlyCost() => Cost;
    
    public virtual void GetSubscriptionDetails()
    {
        Console.WriteLine($"ID: {SubscriptionId}");
        Console.WriteLine($"Услуга: {ServiceName}");
        Console.WriteLine($"Описание: {Description}");
        Console.WriteLine($"Приоритет: {PriorityLevel}");
        Console.WriteLine($"Стоимость: {Cost:C}");
        Console.WriteLine($"Статус: {(IsActive ? "Активна" : "Неактивна")}");
        Console.WriteLine($"Автопродление: {(AutoRenewal ? "Да" : "Нет")}");
        Console.WriteLine($"Всего месяцев подписки: {TotalMonthsSubscribed}");
        Console.WriteLine($"Осталось дней: {GetRemainingDays()}");
    }

    public virtual void DisplayInfo()
    {
        Console.WriteLine($"Базовая информация: {ServiceName} - {Cost:C}");
    }

    // Новый метод с перегрузкой
    public virtual void UpdateSubscription(decimal newCost)
    {
        Cost = newCost;
        Console.WriteLine($"Стоимость подписки обновлена: {newCost:C}");
    }

    public virtual void UpdateSubscription(decimal newCost, string newDescription)
    {
        Cost = newCost;
        Description = newDescription;
        Console.WriteLine($"Подписка полностью обновлена: {newCost:C}, {newDescription}");
    }
}

// Производный класс для онлайн-сервисов
public class OnlineServiceSubscription : Subscription, IEmailNotification
{
    public int MaxUsers { get; set; }
    public bool IsEnterprise { get; set; }
    public string CloudStorageSize { get; set; }
    public string ApiAccessLevel { get; set; } // Новый атрибут
    public int IntegrationCount { get; set; } // Новый атрибут
    public bool TechnicalSupport { get; set; } // Новый атрибут

    public OnlineServiceSubscription(string id, string name, decimal cost, int maxUsers, bool isEnterprise, string storage)
        : base(id, name, cost)
    {
        MaxUsers = maxUsers;
        IsEnterprise = isEnterprise;
        CloudStorageSize = storage;
        ApiAccessLevel = "Basic";
        IntegrationCount = 1;
        TechnicalSupport = false;
    }

    // Новые методы
    public void UpgradeToEnterprise()
    {
        if (!IsEnterprise)
        {
            IsEnterprise = true;
            Cost *= 1.5m;
            ApiAccessLevel = "Full";
            TechnicalSupport = true;
            Console.WriteLine("Обновлено до Enterprise-версии");
        }
    }

    public void AddIntegration()
    {
        IntegrationCount++;
        Console.WriteLine($"Добавлена интеграция. Всего интеграций: {IntegrationCount}");
    }

    public void SetApiAccessLevel(string level)
    {
        ApiAccessLevel = level;
        Console.WriteLine($"Уровень API доступа установлен: {level}");
    }

    // Переопределение методов (полиморфизм)
    public override void ExtendSubscription(int months = 1)
    {
        if (IsEnterprise && months >= 6)
        {
            Console.WriteLine("Специальное предложение для Enterprise: +1 месяц бесплатно!");
            months += 1;
        }
        base.ExtendSubscription(months);
    }

    public override decimal CalculateMonthlyCost()
    {
        decimal total = Cost + (Cost * 0.1m * (MaxUsers - 1));
        total = IsEnterprise ? total * 1.2m : total;
        total += IntegrationCount * 50; // Доплата за интеграции
        return total;
    }

    public override void GetSubscriptionDetails()
    {
        base.GetSubscriptionDetails();
        Console.WriteLine($"Максимум пользователей: {MaxUsers}");
        Console.WriteLine($"Тип: {(IsEnterprise ? "Enterprise" : "Standard")}");
        Console.WriteLine($"Хранилище: {CloudStorageSize}");
        Console.WriteLine($"Уровень API: {ApiAccessLevel}");
        Console.WriteLine($"Количество интеграций: {IntegrationCount}");
        Console.WriteLine($"Техподдержка: {(TechnicalSupport ? "Да" : "Нет")}");
        Console.WriteLine($"Итоговая стоимость: {CalculateMonthlyCost():C}");
    }

    // Реализация интерфейса
    public void SendRenewalReminder()
    {
        int remainingDays = GetRemainingDays();
        if (remainingDays <= 5)
        {
            Console.WriteLine($"СРОЧНОЕ НАПОМИНАНИЕ: подписка истекает через {remainingDays} дней!");
            if (AutoRenewal)
                Console.WriteLine("Будет автоматически продлена");
        }
    }

    // Перегрузка метода
    public void SendRenewalReminder(string customMessage)
    {
        Console.WriteLine($"Персонализированное напоминание: {customMessage}");
    }
}

// Производный класс для стриминговых сервисов
public class StreamingSubscription : Subscription, IPremiumContent
{
    public int MaxStreams { get; set; }
    public List<string> ExclusiveContent { get; set; }
    public bool SupportsHDR { get; set; }
    public string VideoQuality { get; set; } // Новый атрибут
    public int OfflineViewingHours { get; set; } // Новый атрибут
    public List<string> SupportedDevices { get; set; } // Новый атрибут

    public StreamingSubscription(string id, string name, decimal cost, int maxStreams, bool hdr)
        : base(id, name, cost)
    {
        MaxStreams = maxStreams;
        SupportsHDR = hdr;
        ExclusiveContent = new List<string> { "Эксклюзивный фильм", "Закулисные материалы" };
        VideoQuality = "1080p";
        OfflineViewingHours = 48;
        SupportedDevices = new List<string> { "Smart TV", "Phone", "Tablet" };
    }

    // Новые методы
    public void EnableHDR()
    {
        if (SupportsHDR)
        {
            VideoQuality = "4K HDR";
            Console.WriteLine("HDR-качество включено");
        }
        else
            Console.WriteLine("HDR не поддерживается");
    }

    public void AddSupportedDevice(string device)
    {
        SupportedDevices.Add(device);
        Console.WriteLine($"Добавлено поддерживаемое устройство: {device}");
    }

    public void DownloadForOffline(int hours)
    {
        if (hours <= OfflineViewingHours)
        {
            OfflineViewingHours -= hours;
            Console.WriteLine($"Загружено для оффлайн-просмотра на {hours} часов. Осталось: {OfflineViewingHours} часов");
        }
        else
        {
            Console.WriteLine($"Недостаточно оффлайн-часов. Доступно: {OfflineViewingHours}");
        }
    }

    // Переопределение методов
    public override void ExtendSubscription(int months = 1)
    {
        if (months >= 3)
        {
            Console.WriteLine("Специальное предложение! Скидка 15%!");
            decimal discount = Cost * months * 0.15m;
            Console.WriteLine($"Скидка составляет: {discount:C}");
        }
        base.ExtendSubscription(months);
    }

    public override void DisplayInfo()
    {
        Console.WriteLine($"Стриминг сервис: {ServiceName} - {MaxStreams} одновременных потоков");
    }

    // Реализация интерфейса
    public void AccessPremiumFeature()
    {
        Console.WriteLine("Доступ к премиум-контенту: " + string.Join(", ", ExclusiveContent));
        Console.WriteLine($"Доступное качество: {VideoQuality}");
    }

    public override void GetSubscriptionDetails()
    {
        base.GetSubscriptionDetails();
        Console.WriteLine($"Максимум потоков: {MaxStreams}");
        Console.WriteLine($"HDR: {(SupportsHDR ? "Да" : "Нет")}");
        Console.WriteLine($"Качество видео: {VideoQuality}");
        Console.WriteLine($"Оффлайн-просмотр (часов): {OfflineViewingHours}");
        Console.WriteLine($"Поддерживаемые устройства: {string.Join(", ", SupportedDevices)}");
    }
}

// Многоуровневое наследование
public class PremiumStreamingSubscription : StreamingSubscription
{
    public bool DolbyAtmosSupport { get; set; }
    public int OfflineDownloads { get; set; }
    public bool EarlyAccess { get; set; } // Новый атрибут
    public string AudioQuality { get; set; } // Новый атрибут

    public PremiumStreamingSubscription(string id, string name, decimal cost, int maxStreams, bool hdr, bool atmos, int downloads)
        : base(id, name, cost, maxStreams, hdr)
    {
        DolbyAtmosSupport = atmos;
        OfflineDownloads = downloads;
        EarlyAccess = true;
        AudioQuality = "High-Resolution";
        VideoQuality = "4K Dolby Vision"; // Переопределение атрибута базового класса
    }

    // Новые методы
    public void DownloadOffline()
    {
        if (OfflineDownloads > 0)
        {
            OfflineDownloads--;
            Console.WriteLine($"Загружено для оффлайн-просмотра. Осталось слотов: {OfflineDownloads}");
        }
        else
        {
            Console.WriteLine("Лимит оффлайн-загрузок исчерпан");
        }
    }

    public void GetEarlyAccess()
    {
        if (EarlyAccess)
        {
            Console.WriteLine("Ранний доступ к новым релизам предоставлен!");
        }
    }

    // Переопределение методов
    public override void ExtendSubscription(int months = 1)
    {
        Console.WriteLine("Премиум подписка продлена с бонусными функциями!");
        if (months >= 6)
        {
            EarlyAccess = true;
            Console.WriteLine("Активирован ранний доступ на следующий период!");
        }
        base.ExtendSubscription(months);
    }

    public override decimal CalculateMonthlyCost()
    {
        decimal baseCost = base.CalculateMonthlyCost();
        return baseCost * 1.3m; // Наценка за премиум функции
    }

    public override void GetSubscriptionDetails()
    {
        base.GetSubscriptionDetails();
        Console.WriteLine($"Dolby Atmos: {(DolbyAtmosSupport ? "Да" : "Нет")}");
        Console.WriteLine($"Оффлайн-загрузки: {OfflineDownloads}");
        Console.WriteLine($"Ранний доступ: {(EarlyAccess ? "Да" : "Нет")}");
        Console.WriteLine($"Качество аудио: {AudioQuality}");
        Console.WriteLine($"Премиум стоимость: {CalculateMonthlyCost():C}");
    }
}

        Console.WriteLine("=== Online Service Subscription ===");
        var onlineSub = new OnlineServiceSubscription("ONL123", "Office 365", 100, 5, false, "1TB");
        onlineSub.GetSubscriptionDetails();
        onlineSub.SendRenewalReminder();
        onlineSub.UpgradeToEnterprise();
        onlineSub.AddIntegration();
        onlineSub.ExtendSubscription(6, true);
        onlineSub.GetSubscriptionDetails();
        Console.WriteLine();

        Console.WriteLine("=== Streaming Subscription ===");
        var streamSub = new StreamingSubscription("STR456", "Netflix", 200, 3, true);
        streamSub.GetSubscriptionDetails();
        streamSub.AccessPremiumFeature();
        streamSub.EnableHDR();
        streamSub.AddSupportedDevice("Gaming Console");
        streamSub.DownloadForOffline(24);
        Console.WriteLine();

        Console.WriteLine("=== Premium Streaming Subscription ===");
        var premiumSub = new PremiumStreamingSubscription("PRE789", "Netflix Premium", 300, 4, true, true, 5);
        premiumSub.GetSubscriptionDetails();
        premiumSub.DownloadOffline();
        premiumSub.AccessPremiumFeature();
        premiumSub.GetEarlyAccess();
        Console.WriteLine();

        // Демонстрация полиморфизма
        Console.WriteLine("=== Демонстрация полиморфизма ===");
        List<Subscription> subscriptions = new List<Subscription> { onlineSub, streamSub, premiumSub };
        
        foreach (var sub in subscriptions)
        {
            sub.DisplayInfo(); // Полиморфный вызов
            sub.ExtendSubscription(2); // Полиморфный вызов
            Console.WriteLine("---");
        }

        // Демонстрация generic класса
        Console.WriteLine("=== Демонстрация Generic Manager ===");
        var subscriptionManager = new SubscriptionManager<Subscription>();
        subscriptionManager.AddSubscription(onlineSub);
        subscriptionManager.AddSubscription(streamSub);
        subscriptionManager.AddSubscription(premiumSub);
        
        subscriptionManager.DisplayAllSubscriptions();

        // Менеджер для конкретного типа
        var streamingManager = new SubscriptionManager<StreamingSubscription>();
        streamingManager.AddSubscription(streamSub);
        streamingManager.AddSubscription(premiumSub); // PremiumStreamingSubscription является StreamingSubscription
        streamingManager.DisplayAllSubscriptions();

=== Online Service Subscription ===
ID: ONL123
Услуга: Office 365
Описание: Базовая подписка
Приоритет: 1
Стоимость: ¤100.00
Статус: Активна
Автопродление: Нет
Всего месяцев подписки: 0
Осталось дней: 30
Максимум пользователей: 5
Тип: Standard
Хранилище: 1TB
Уровень API: Basic
Количество интеграций: 1
Техподдержка: Нет
Итоговая стоимость: ¤190.00
Обновлено до Enterprise-версии
Добавлена интеграция. Всего интеграций: 2
Применена скидка 10%: -¤90.00
Специальное предложение для Enterprise: +1 месяц бесплатно!
Подписка Office 365 продлена на 7 месяц(ев)
ID: ONL123
Услуга: Office 365
Описание: Базовая подписка
Приоритет: 1
Стоимость: ¤150.00
Статус: Активна
Автопродление: Нет
Всего месяцев подписки: 7
Осталось дней: 242
Максимум пользователей: 5
Тип: Enterprise
Хранилище: 1TB
Уровень API: Full
Количество интеграций: 2
Техподдержка: Да
Итоговая стоимость: ¤352.00

=== Streaming Subscription ===
ID: STR456
Услуга: Netflix
Описание: Базовая подписка
Приоритет: 1
Стоимость: ¤200.00
Статус: Акти